# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 20;  10
2. Алматы Nan; Шымкент 3; Астана Nan
3. (2, 2)
4. series, dataframe
5. [1, 2] 5

In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [4]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [5]:
# 1
print("колво строк и столбцов", df.shape)
print("типы данных столбцов:", df.dtypes)

print("пропусков у delivery days -", df["delivery_days"].isna().sum())

print(f"разных значений у столбца city - {df["city"].nunique()}")
print(f"разных значений у столбца category - {df["category"].nunique()}")
print(f"разных значений у столбца status - {df["status"].nunique()}")
print(f"разных значений у столбца customer_id - {df["customer_id"].nunique()}")

# delivery days имеет 160 пропусков
# столбец city включает себя названия одних и тех же городов с разным написанием
# риск дубликата заказов

колво строк и столбцов (5463, 8)
типы данных столбцов: order_id           int64
order_date           str
customer_id          str
city                 str
category             str
amount           float64
delivery_days    float64
status               str
dtype: object
пропусков у delivery days - 160
разных значений у столбца city - 14
разных значений у столбца category - 5
разных значений у столбца status - 2
разных значений у столбца customer_id - 2003


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [6]:
# 2
mask = ( df["status"] == "оплачен" ) & ( df["customer_id"] != "TEST" )
filtered = df.loc[mask, "amount"]

print ("Среднее", filtered.median())
print ("Медиана", round ( filtered.mean() , 1) )
print ("25й перцентиль", filtered.quantile(0.25))
print ("75 перцентиль", filtered.quantile(0.75))   
#Медиана лучше характеризует так как она меньше зависит от экстремальных значений (outliers) 

Среднее 10300.0
Медиана 14672.4
25й перцентиль 5880.0
75 перцентиль 18690.0


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [26]:
# 3
mask = (df["status"] == "отменён")

shymkent = df["city"].isin(["Шымкент", "шымкент", "Shymkent"])
almaty = df["city"].isin(["Алматы", " Алматы", "Almaty", "алматы", "АЛМАТЫ"])

total_rate = mask.mean()
rate_shymkent = mask[shymkent].mean()
rate_almaty = mask[almaty].mean()

print (f"Доля отмененных заказов по всем заказам {total_rate:.2f}")
print (f"Доля отмененных заказов в Шымкенте {rate_shymkent:.2f}")
print (f"Доля отмененных заказов Алматы {rate_almaty:.2f}")
print (f"Доля отмененных заказов в Шымкенте больше чем в Алматы в {(rate_shymkent / rate_almaty):.2f} раза")

Доля отмененных заказов по всем заказам 0.10
Доля отмененных заказов в Шымкенте 0.20
Доля отмененных заказов Алматы 0.08
Доля отмененных заказов в Шымкенте больше чем в Алматы в 2.59 раза


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [45]:
# 4
mask = (df["status"] == "оплачен") & (df["customer_id"] != "TEST")

shymkent = df["city"].isin(["Шымкент", "шымкент", "Shymkent"])
almaty = df["city"].isin(["Алматы", " Алматы", "Almaty", "алматы", "АЛМАТЫ"])

sep = df["order_date"] < "2026-10"
oct = df["order_date"] >= "2026-10"

shymkent_sep = df.loc[mask & shymkent & sep, "amount"].sum()
shymkent_oct = df.loc[mask & shymkent & oct, "amount"].sum()

almaty_sep = df.loc[mask & almaty & sep, "amount"].sum()
almaty_oct = df.loc[mask & almaty & oct, "amount"].sum()

shymkent_change = ( (shymkent_oct - shymkent_sep) / shymkent_sep ) * 100
almaty_change = ( (almaty_oct - almaty_sep) / almaty_sep ) * 100

print (f"Выручка Шымкента за сентябрь {shymkent_sep}, октябрь {shymkent_oct}")
print (f"Выручка Шымкента снизилась на, { round(abs(shymkent_change),2 ) } %") 

print (f"Выручка Алматы за сентябрь {almaty_sep}, октябрь {almaty_oct}")
print (f"Выручка Шымкента снизилась на, { round(abs(almaty_change),2 ) } %") 

print (f"Падение сильнее в Шымкенте в { round (shymkent_change / almaty_change, 2) } раза")

Выручка Шымкента за сентябрь 4949800.0, октябрь 4044010.0
Выручка Шымкента снизилась на, 18.3 %
Выручка Алматы за сентябрь 15523940.0, октябрь 14643020.0
Выручка Шымкента снизилась на, 5.67 %
Падение сильнее в Шымкенте в 3.22 раза


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [ ]:
# 5
# Отсутствуют скобки разделяющие условия city == шымкент и статус == оплачен
# Фильтруется только один вариант написания Шымкента когда их в таблице три варианта
cancel_sh = df[ df["city"].isin(["Шымкент", "шымкент", "Shymkent"]) & (df["status"] == "отменён") ]
print("Количество отменённых заказов Шымкента:", len(cancel_sh))

Количество отменённых заказов Шымкента: 164


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [60]:
# 6
customer_count = df["customer_id"].value_counts()

top_customer = customer_count.index[0]
total_orders = customer_count.iloc[0]

print("Самый частый клиент:", top_customer)
print("Количество заказов:", total_orders)

orders = df.loc[
    df["customer_id"] == top_customer,
    ["order_id", "order_date", "city", "amount", "status"]
]
print(orders)

duplicates = orders["order_id"].duplicated().sum()

print("Количество повторных order_id:", duplicates)

print(orders [ orders["order_id"].duplicated(keep = False) ] )

# Дубликаты order_id завышают количество заказов если один и тот же заказ учитывался пару раз

Самый частый клиент: C01818
Количество заказов: 11
      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен
Количество повторных order_id: 1
      order_id        order_date    city  amount   status
1371    101504  2026-09-15 21:00  Алматы  8890.0  оплачен
1375    101504  2026-09-15 21:00  Алматы  8890.0  оплаче

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [ ]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8